# 🏛️ Stage 1: Philippine Statutory Retrieval & Dense Pre-computation Pipeline
**Thesis Title**: *A Coarse-to-Fine Semantic Conflict Detection System for Ex-Ante Davao City Ordinances Using Information Retrieval and Natural Language Inference*
**Authors**: Ralph Paolo Dulce & Yahyah Odin (Ateneo de Davao University)

---
### 📋 Overview
This notebook implements and evaluates **Stage 1 (Statutory Candidate Retrieval)** of our two-stage architecture:
1. **RRL & Theoretical Alignment**:
   - Implements **COLIEE Task 3 (Statute Law Retrieval)** as the computational realization of the *Magtajas Doctrine* (Pillar 1).
   - Adapts **Team JNLP's Hybrid Lexical-Neural Search** and **Weighted Sum Aggregation**.
   - Incorporates **AIIR Lab's BM25 Lexical Preprocessing**.
2. **Decoupled Corpus Pre-computation**:
   - Encodes the **25,432 national statutory corpus** into static dense vectors using a pre-trained Transformer Bi-Encoder (`BAAI/bge-m3` or `sentence-transformers/all-mpnet-base-v2`).
   - Decoupling heavy GPU pre-computation offline allows live local queries on consumer hardware to execute in milliseconds via dot-product matrix multiplication.
3. **Empirical Evaluation Benchmark**:
   - Evaluates retrieval across all **350 Ground Truth query clauses**.
   - Compares: **Pure BM25** vs. **Pure Dense** vs. **Hybrid Fusion (RRF / Weighted Sum)** vs. **Safeguards** (Statutory Hierarchy & BERTopic Domain filtering).
   - Computes **Recall@10, Recall@30, Recall@50**, **MRR**, and **Difficulty Tier breakdowns** (Tier 1 vs. Tier 2 vs. Tier 3) ready for Thesis Chapter 4.

In [ ]:
# Cell 1: Environment Setup & High-Performance Dependencies
# Run this cell in Google Colab (Select GPU Runtime: Runtime -> Change runtime type -> T4 GPU)
!pip install -q sentence-transformers rank-bm25 pandas numpy scikit-learn tqdm accelerate

import os
import sys
import re
import json
import time
from typing import List, Dict, Any, Tuple
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Running on CPU. For fast pre-computation, please enable GPU runtime in Colab.")

In [ ]:
# Cell 2: Ingest the 25,432 National Statutory Corpus & 350 Ground Truth Benchmark
# Clone repository or ensure data files are present
if not os.path.exists("corpus/categorized_national_laws.jsonl"):
    if os.path.exists("thesis-repo"):
        os.chdir("thesis-repo")
    else:
        !git clone https://github.com/yyaahhzxc/thesis-repo.git
        os.chdir("thesis-repo")

corpus_path = "corpus/categorized_national_laws.jsonl"
gt_path = "data/ground_truth_350.jsonl"

print(f"Loading corpus from: {corpus_path}")
corpus_records = []
with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            corpus_records.append(json.loads(line))
print(f"✅ Successfully loaded {len(corpus_records):,} national legal statutes.")

print(f"Loading ground truth from: {gt_path}")
gt_records = []
with open(gt_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            gt_records.append(json.loads(line))
print(f"✅ Successfully loaded {len(gt_records)} ground truth test queries.")

In [ ]:
# Cell 3: Hierarchical Passage Construction (Section 3.2.3.2)
# Formula: x_chunk = [Statute Title, Section S_i: Catchline] || t_verbatim

doc_ids = [r['law_id'] for r in corpus_records]
law_numbers = [r.get('law_number', '') for r in corpus_records]
long_titles = [r.get('long_title', '') for r in corpus_records]
categories = [r.get('category', '') for r in corpus_records]
topic_ids = np.array([r.get('topic_id', -1) for r in corpus_records], dtype=int)
is_primary_mask = np.array([
    any(k in c.lower() for k in ['republic', 'batas', 'act', 'commonwealth'])
    for c in categories
], dtype=bool)

# Format search passages with hierarchical headers
passages = []
for r in corpus_records:
    ft = r.get('full_text') or r.get('searchable_doc') or ''
    body_snip = ft[:3500]
    # Hierarchical representation: Title + Section Header + Verbatim excerpt
    p = f"Statute: {r.get('law_number', '')} - {r.get('long_title', '')}. Excerpt: {body_snip}"
    passages.append(p)

print(f"Sample structured passage (first 300 chars):\n{passages[0][:300]}...")

In [ ]:
# Cell 4: Offline Dense Pre-Computation (SentenceTransformer GPU Encoding)
# Pre-computes and normalizes vectors for all 25,432 statutes
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"  # or "BAAI/bge-m3"
print(f"Loading dense embedding model: {MODEL_NAME} on {device}...")
dense_model = SentenceTransformer(MODEL_NAME, device=device)

embeddings_file = "data/statute_embeddings.npy"
if os.path.exists(embeddings_file):
    print(f"Loading existing pre-computed embeddings from {embeddings_file}...")
    statute_embeddings = np.load(embeddings_file)
    print(f"✅ Loaded embeddings with shape: {statute_embeddings.shape}")
else:
    print(f"⚡ Computing dense embeddings for all {len(passages):,} statutes (Takes ~2-3 mins on T4 GPU)...")
    t0 = time.time()
    statute_embeddings = dense_model.encode(
        passages,
        batch_size=64 if device == 'cuda' else 16,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    t1 = time.time()
    print(f"✅ Encoded {len(passages):,} statutes in {t1 - t0:.2f} seconds ({len(passages)/(t1-t0):.1f} docs/sec)!")
    
    os.makedirs("data", exist_ok=True)
    np.save(embeddings_file, statute_embeddings)
    print(f"💾 Saved embeddings to {embeddings_file} ({os.path.getsize(embeddings_file) / (1024**2):.1f} MB).")

In [ ]:
# Cell 5: Build BM25 Inverted Index (AIIR Lab Lexical Baseline)
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> List[str]:
    return re.findall(r'[a-zA-Z0-9]+', text.lower())

print("Tokenizing corpus for BM25...")
t0 = time.time()
tokenized_corpus = [tokenize(p) for p in passages]
bm25_index = BM25Okapi(tokenized_corpus)
t1 = time.time()
print(f"✅ BM25 inverted index constructed in {t1 - t0:.2f} seconds.")

In [ ]:
# Cell 6: Benchmark Evaluation Framework (BM25 vs Dense vs Hybrid Fusion)
# Resolves ground truth queries and measures Recall@k, MRR, Latency, and Tier breakdown

TITLE_TO_LAW_ID = {
    'Republic Act No. 10121 (Philippine Disaster Risk Reduction and Management Act of 2010)': 'ra_10121_2010',
    'Republic Act No. 10591 (Comprehensive Firearms and Ammunition Regulation Act)': 'ra_10591_2013',
    'Republic Act No. 10931 (Universal Access to Quality Tertiary Education Act)': 'ra_10931_2017',
    'Republic Act No. 11032 (Ease of Doing Business and Efficient Government Service Delivery Act of 2018)': 'ra_11032_2018',
    'Republic Act No. 11223 (Universal Health Care Act)': 'ra_11223_2019',
    'Republic Act No. 11314 (Student Fare Discount Act)': 'ra_11314_2019',
    'Republic Act No. 11332 (Mandatory Reporting of Notifiable Diseases and Health Events of Public Health Concern Act)': 'ra_11332_2019',
    'Republic Act No. 4136 (Land Transportation and Traffic Code)': 'ra_4136_1964',
    'Republic Act No. 7160 (The Local Government Code of 1991)': 'ra_7160_1991',
    'Republic Act No. 7581 (The Price Act)': 'ra_7581_1992',
    'Republic Act No. 7925 (Public Telecommunications Policy Act of 1995)': 'ra_7925_1995',
    'Republic Act No. 7942 (Philippine Mining Act of 1995)': 'ra_7942_1995',
    'Republic Act No. 8550 (The Philippine Fisheries Code of 1998)': 'ra_8550_1998',
    'Republic Act No. 9136 (Electric Power Industry Reform Act of 2001)': 'ra_9136_2001',
    'Republic Act No. 9165 (Comprehensive Dangerous Drugs Act of 2002)': 'ra_9165_2002',
    'Republic Act No. 9211 (Tobacco Regulation Act of 2003)': 'ra_9211_2003',
    'Republic Act No. 9344 (Juvenile Justice and Welfare Act of 2006)': 'ra_9344_2006',
}

id_to_idx = {lid: i for i, lid in enumerate(doc_ids)}

def evaluate_retrieval_system(mode: str, alpha: float = 0.5, use_hierarchy_safeguard: bool = True):
    latencies = []
    tier_ranks = {
        "Overall": [],
        "Tier 1: Surface & Quantitative": [],
        "Tier 2: Preemption & Carve-Outs": [],
        "Tier 3: Latent & Paraphrastic": []
    }
    
    k_vals = [5, 10, 20, 30, 50]
    
    for item in gt_records:
        query = item['ordinance_hypothesis']['hypothesis_text']
        title = item['national_premise']['statute_title']
        tier = item.get('difficulty_tier', 'Unknown')
        
        target_id = TITLE_TO_LAW_ID.get(title)
        if not target_id or target_id not in id_to_idx:
            continue
        target_idx = id_to_idx[target_id]
        
        t0 = time.perf_counter()
        
        # Score calculation based on mode
        if mode == "bm25":
            q_tokens = tokenize(query)
            scores = np.array(bm25_index.get_scores(q_tokens), dtype=np.float32)
        elif mode == "dense":
            q_emb = dense_model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
            scores = np.dot(statute_embeddings, q_emb)
        elif mode == "hybrid":
            # Normalize BM25 and Dense scores to [0, 1] and perform Weighted Sum Aggregation (Team JNLP)
            q_tokens = tokenize(query)
            bm25_s = np.array(bm25_index.get_scores(q_tokens), dtype=np.float32)
            if bm25_s.max() > bm25_s.min():
                bm25_norm = (bm25_s - bm25_s.min()) / (bm25_s.max() - bm25_s.min())
            else:
                bm25_norm = bm25_s
                
            q_emb = dense_model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
            dense_s = np.dot(statute_embeddings, q_emb)
            dense_norm = (dense_s + 1.0) / 2.0  # Cosine [-1, 1] to [0, 1]
            
            scores = alpha * dense_norm + (1.0 - alpha) * bm25_norm
            
        if use_hierarchy_safeguard:
            scores[~is_primary_mask] = -1e9
            
        target_score = scores[target_idx]
        rank = int(np.sum(scores > target_score)) + 1
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000.0)
        tier_ranks["Overall"].append(rank)
        if tier in tier_ranks:
            tier_ranks[tier].append(rank)
            
    results = {}
    for group, ranks in tier_ranks.items():
        if len(ranks) == 0:
            continue
        recalls = {f"Recall@{k}": round(float(np.mean([1.0 if r <= k else 0.0 for r in ranks])) * 100.0, 1) for k in k_vals}
        mrr50 = round(float(np.mean([1.0 / r if r <= 50 else 0.0 for r in ranks])), 4)
        results[group] = {**recalls, "MRR@50": mrr50, "N": len(ranks)}
        
    results["latency_mean_ms"] = round(float(np.mean(latencies)), 1)
    return results

In [ ]:
# Cell 7: Execute Comparative Benchmark & Display Thesis Results Table
print("Running Stage 1 Empirical Comparisons across 350 queries...")

benchmark_runs = {
    "1. Pure BM25 (AIIR Lab)": evaluate_retrieval_system(mode="bm25", use_hierarchy_safeguard=True),
    "2. Pure Dense (all-mpnet-base-v2)": evaluate_retrieval_system(mode="dense", use_hierarchy_safeguard=True),
    "3. Hybrid Fusion (JNLP Weighted Sum α=0.5)": evaluate_retrieval_system(mode="hybrid", alpha=0.5, use_hierarchy_safeguard=True),
    "4. Hybrid Fusion (Dense Biased α=0.7)": evaluate_retrieval_system(mode="hybrid", alpha=0.7, use_hierarchy_safeguard=True),
}

# Build Master Comparison Table
rows = []
for name, res in benchmark_runs.items():
    ov = res["Overall"]
    rows.append({
        "Model Configuration": name,
        "Recall@10": f"{ov['Recall@10']}%",
        "Recall@30": f"{ov['Recall@30']}%",
        "Recall@50": f"{ov['Recall@50']}%",
        "MRR@50": ov["MRR@50"],
        "Tier 1 (R@50)": f"{res['Tier 1: Surface & Quantitative']['Recall@50']}%",
        "Tier 2 (R@50)": f"{res['Tier 2: Preemption & Carve-Outs']['Recall@50']}%",
        "Tier 3 (R@50)": f"{res['Tier 3: Latent & Paraphrastic']['Recall@50']}%",
        "Latency (ms)": f"{res['latency_mean_ms']} ms"
    })

df_results = pd.DataFrame(rows)
display(df_results)

# Export to LaTeX table format
print("\n--- LaTeX Table Output (Ready for Chapter 4) ---")
print(df_results.to_latex(index=False))